In [1]:
import os

DATA_PATH = './data/'
MODEL_PATH = './models/'

try:
    from google.colab import drive

    drive.mount('/content/drive')

    DATA_PATH = '/content/drive/MyDrive/data/'
    MODEL_PATH = '/content/drive/MyDrive/models/'
    DIARY_PATH = '/content/drive/MyDrive/diaries/'
    import sys
    sys.path.insert(0, '/content/drive/MyDrive/')
    print("Running in Google Colab")
    print("Drive path inserted")
except ImportError:
    print("Running locally")


Running locally


Global Seed

In [2]:
import numpy as np
import torch
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

Data preprocessing


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import random

print(f"seed used : {SEED}")

architectures = []
hidden_layers_num = []
hidden_layers_sizes = []
hyperparams = []
scores = []

def hyper_params_str(bs, eps, pt, l, wc, dr):
    return f"batch size : {bs}, epochs {eps}, early stopping patience: {pt}, lr : {l}, l2 reg: {wc}, dropout_rate:{dr}"


processed_datasets = {
    "X_train": None,
    "X_val": None,
    "y_train": None,
    "y_val": None
}


for psd in processed_datasets:
  dataset = Path(DATA_PATH + psd + ".pt")
  if dataset.exists():
    processed_datasets[psd] = torch.load(str(dataset), weights_only=True)
  else:
    print(f"dataset {psd} not found")

classes_ = None
if Path(DATA_PATH + "classes.pt").exists():
  classes_ = torch.load(str(DATA_PATH + "classes.pt"), weights_only=False)


if None not in processed_datasets.values() and classes_ is not None:
  X_train, X_val, y_train, y_val = processed_datasets.values()
  print(f"all preprocessed datasets are loaded from {DATA_PATH}")

else:
  print("dataset processing...")
  from tools.preprocessor import NLProcessor

  df = pd.concat([
      pd.read_csv(DATA_PATH + 'intents_train.csv'),
      pd.read_csv(DATA_PATH + 'bike_shop_intents.csv')
      # oos_df
  ], ignore_index=True)

  X_train, X_val, y_train, y_val = train_test_split(
      df['text'], df['intent'], test_size=0.15, random_state=SEED, stratify=df['intent']
  )

  del df

  le = LabelEncoder()
  processor = NLProcessor(use_stopwords=True)

  X_train = torch.from_numpy(processor.transform(X_train).astype(np.float32))
  X_val = torch.from_numpy(processor.transform(X_val).astype(np.float32))

  y_train = torch.from_numpy(le.fit_transform(y_train)).long()
  y_val = torch.from_numpy(le.transform(y_val)).long()

  y_val = np.delete(y_val, processor.empty_indices)

  classes_ = le.classes_
  torch.save(X_train, f"{DATA_PATH}X_train.pt")
  torch.save(X_val, f"{DATA_PATH}X_val.pt")
  torch.save(y_train, f"{DATA_PATH}y_train.pt")
  torch.save(y_val, f"{DATA_PATH}y_val.pt")
  torch.save(classes_, f"{DATA_PATH}classes.pt")

  print(f"preprocessed datasets are saved at {DATA_PATH}")


print(f"X train shape : {X_train.shape}")
print(f"X val shape : {X_val.shape}")
print(f"Y train shape : {y_train.shape}")
print(f"Y val shape : {y_val.shape}")
print(f"classes : {classes_}")


seed used : 42
all preprocessed datasets are loaded from ./data/
X train shape : torch.Size([650, 8, 300])
X val shape : torch.Size([115, 8, 300])
Y train shape : torch.Size([650])
Y val shape : torch.Size([115])
classes : ['Bike Types' 'Cost Estimation' 'Hours' 'Make Appointment' 'Return Policy'
 'Trade-in Options' 'Welcome Intent']


Prepare Training Dataloader

In [4]:

from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
from tools.train import train
from tools.earlystopping import EarlyStopping

BATCH_SIZE = 32

dataloader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=False
)
print(dataloader.dataset)

RNN Solution

In [5]:
from tools.rnn import RNN

dropout_rate = 0.2
patience = 10
lr = 0.001
weight_decay = 1e-4
scheduler_factor = 0.5
num_epochs = 120
hidden_size = 256
num_layers = 2
label_smoothing = 0.1


model = RNN(
    input_size=X_train.shape[2],
    hidden_size=hidden_size,
    num_layers=num_layers,
    num_classes=len(classes_),
    dropout_rate=dropout_rate
)

criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
earlystopping = EarlyStopping(patience=patience)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=scheduler_factor, patience=patience
)

for epoch in range(num_epochs):

    model.train()
    avg_train_loss, train_acc = train(model, criterion, optimizer, dataloader)

    print(f"epoch {epoch + 1}/{num_epochs} | ", end='')
    print(f"train loss : {avg_train_loss:.3f} | train acc : {train_acc:.2f}")

    model.eval()

    val_loss, val_acc = EarlyStopping.compute_val_metrics(model, criterion, X_val, y_val)
    scheduler.step(val_loss)
    print(f"val loss : {val_loss:.3f} | val acc : {val_acc:.3f}")

    if earlystopping(val_loss, val_acc, model):
        break


torch.save(model, f"{MODEL_PATH}rnn.pkl")


architectures.append(model.__class__.__name__)
hyperparams.append(hyper_params_str(BATCH_SIZE, num_epochs, patience, lr, weight_decay, dropout_rate))
scores.append(earlystopping.best_accuracy)
hidden_layers_num .append(num_layers)
hidden_layers_sizes.append(hidden_size)

epoch 1/120 | train loss : 1.696 | train acc : 0.33
val loss : 0.011 | val acc : 0.574
epoch 2/120 | train loss : 1.175 | train acc : 0.64
val loss : 0.010 | val acc : 0.696
epoch 3/120 | train loss : 0.933 | train acc : 0.78
val loss : 0.010 | val acc : 0.713
epoch 4/120 | train loss : 0.819 | train acc : 0.84
val loss : 0.008 | val acc : 0.809
epoch 5/120 | train loss : 0.750 | train acc : 0.88
val loss : 0.007 | val acc : 0.800
epoch 6/120 | train loss : 0.659 | train acc : 0.92
val loss : 0.009 | val acc : 0.739
epoch 7/120 | train loss : 0.674 | train acc : 0.91
val loss : 0.009 | val acc : 0.774
epoch 8/120 | train loss : 0.645 | train acc : 0.92
val loss : 0.009 | val acc : 0.817
epoch 9/120 | train loss : 0.672 | train acc : 0.91
val loss : 0.008 | val acc : 0.826
epoch 10/120 | train loss : 0.690 | train acc : 0.90
val loss : 0.008 | val acc : 0.800
epoch 11/120 | train loss : 0.644 | train acc : 0.92
val loss : 0.008 | val acc : 0.826
epoch 12/120 | train loss : 0.677 | train

LSTM Solution

In [6]:
from tools.lstm import LSTM

dropout_rate = 0.1
patience = 12
lr = 0.001
weight_decay = 0.0001
scheduler_factor = 0.5
num_epochs = 100
hidden_size = 256
num_layers = 2
label_smoothing = 0.2


model = LSTM(
    input_size=X_train.shape[2],
    hidden_size=hidden_size,
    num_layers=num_layers,
    num_classes=len(classes_),
    dropout_rate=dropout_rate
)

criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
earlystopping = EarlyStopping(patience=patience)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=scheduler_factor, patience=patience
)

for epoch in range(num_epochs):

    model.train()
    avg_train_loss, train_acc = train(model, criterion, optimizer, dataloader)

    print(f"epoch {epoch + 1}/{num_epochs} | ", end='')
    print(f"train loss : {avg_train_loss:.3f} | train acc : {train_acc:.3f}")

    model.eval()
    val_loss, val_acc = EarlyStopping.compute_val_metrics(model, criterion, X_val, y_val)
    scheduler.step(val_loss)
    print(f"val loss : {val_loss:.3f} | val acc : {val_acc:.3f}")

    if earlystopping(val_loss ,val_acc, model):
        break

print(f"LSTM best accuracy reach in validation set : {earlystopping.best_accuracy:.3f}")
torch.save(model, f"{MODEL_PATH}lstm.pkl")
print(f"lstm model saved as {MODEL_PATH}lstm.pkl")


architectures.append(model.__class__.__name__)
hyperparams.append(hyper_params_str(BATCH_SIZE, num_epochs, patience, lr, weight_decay, dropout_rate))
scores.append(earlystopping.best_accuracy)
hidden_layers_num .append(num_layers)
hidden_layers_sizes.append(hidden_size)

epoch 1/100 | train loss : 1.892 | train acc : 0.225
val loss : 0.016 | val acc : 0.357
epoch 2/100 | train loss : 1.639 | train acc : 0.434
val loss : 0.013 | val acc : 0.609
epoch 3/100 | train loss : 1.234 | train acc : 0.712
val loss : 0.011 | val acc : 0.704
epoch 4/100 | train loss : 1.029 | train acc : 0.865
val loss : 0.010 | val acc : 0.791
epoch 5/100 | train loss : 0.959 | train acc : 0.897
val loss : 0.009 | val acc : 0.870
epoch 6/100 | train loss : 0.909 | train acc : 0.918
val loss : 0.009 | val acc : 0.861
epoch 7/100 | train loss : 0.871 | train acc : 0.946
val loss : 0.009 | val acc : 0.870
epoch 8/100 | train loss : 0.854 | train acc : 0.946
val loss : 0.010 | val acc : 0.826
epoch 9/100 | train loss : 0.866 | train acc : 0.945
val loss : 0.009 | val acc : 0.870
epoch 10/100 | train loss : 0.845 | train acc : 0.955
val loss : 0.009 | val acc : 0.870
epoch 11/100 | train loss : 0.850 | train acc : 0.960
val loss : 0.009 | val acc : 0.861
epoch 12/100 | train loss : 0.

CNN Solution

In [7]:
from tools.cnn import TextCNN
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
from tools.train import train
from tools.earlystopping import EarlyStopping

import numpy as np
import torch
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

BATCH_SIZE = 64
dropout_rate = 0.1
patience = 12
lr = 0.001
weight_decay = 1e-4
scheduler_factor = 0.5
num_epochs = 120
kernel_sizes=[3, 5, 7]
label_smoothing = 0.05
out_channels = 256


X_train_ = X_train.permute(0, 2, 1)
X_val_   = X_val.permute(0, 2, 1)

batch_size, embed_dim, seq_len = X_train_.shape

dataloader = DataLoader(
    TensorDataset(X_train_, y_train),
    batch_size=BATCH_SIZE
)

model = TextCNN(
    embed_dim=embed_dim,
    out_channels=out_channels,
    output_size=len(classes_),
    kernel_sizes=kernel_sizes,
    dropout_rate=dropout_rate,
    classes_=classes_
)

criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=scheduler_factor, patience=patience
)
earlystopping = EarlyStopping(patience=patience)

for epoch in range(num_epochs):

    model.train()
    avg_train_loss, train_acc = train(model, criterion, optimizer, dataloader)

    print(f"epoch {epoch + 1}/{num_epochs} | ", end='')
    print(f"train loss : {avg_train_loss:.3f} | train acc : {train_acc:.3f}")

    model.eval()
    val_loss, val_acc = EarlyStopping.compute_val_metrics(model, criterion, X_val_, y_val)
    scheduler.step(val_loss)
    print(f"val loss : {val_loss:.3f} | val acc : {val_acc:.3f}")

    if earlystopping(val_loss, val_acc, model):
        break

print(f"CNN best accuracy reach in validation set : {earlystopping.best_accuracy}")

torch.save(model, f"{MODEL_PATH}cnn.pkl")
print(f"text cnn saved as {MODEL_PATH}cnn.pkl")

architectures.append(model.__class__.__name__)
hyperparams.append(hyper_params_str(BATCH_SIZE, num_epochs, patience, lr, weight_decay, dropout_rate))
scores.append(earlystopping.best_accuracy)
hidden_layers_num .append(num_layers)
hidden_layers_sizes.append(hidden_size)

epoch 1/120 | train loss : 1.366 | train acc : 0.540
val loss : 0.014 | val acc : 0.678
epoch 2/120 | train loss : 0.581 | train acc : 0.878
val loss : 0.010 | val acc : 0.861
epoch 3/120 | train loss : 0.483 | train acc : 0.938
val loss : 0.007 | val acc : 0.904
epoch 4/120 | train loss : 0.376 | train acc : 0.969
val loss : 0.005 | val acc : 0.922
epoch 5/120 | train loss : 0.321 | train acc : 0.988
val loss : 0.004 | val acc : 0.922
epoch 6/120 | train loss : 0.304 | train acc : 0.991
val loss : 0.004 | val acc : 0.930
epoch 7/120 | train loss : 0.303 | train acc : 0.986
val loss : 0.004 | val acc : 0.948
epoch 8/120 | train loss : 0.287 | train acc : 0.992
val loss : 0.004 | val acc : 0.939
epoch 9/120 | train loss : 0.279 | train acc : 0.994
val loss : 0.003 | val acc : 0.939
epoch 10/120 | train loss : 0.281 | train acc : 0.988
val loss : 0.004 | val acc : 0.957
epoch 11/120 | train loss : 0.280 | train acc : 0.991
val loss : 0.004 | val acc : 0.930
epoch 12/120 | train loss : 0.

BERT Solution

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import BertTokenizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset
from tools.earlystopping import EarlyStopping
from tools.bert import IntentClassifier
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE        = 32
dropout_rate      = 0.1
patience          = 10
bert_lr           = 2e-5
head_lr           = 1e-3
weight_decay      = 1e-4
scheduler_factor  = 0.5
num_epochs        = 100
max_length        = 128
hidden_layer_size = 128
val_size          = 0.15
label_smoothing   = 0.1
BERT_MODEL_NAME   = "bert-base-uncased"


df = pd.concat([
    pd.read_csv(DATA_PATH + 'intents_train.csv'),
    pd.read_csv(DATA_PATH + 'bike_shop_intents.csv')
], ignore_index=True)

print(f"Original df shape: {df.shape}")
print(f"Intent distribution:\n{df['intent'].value_counts()}")


x_train, x_val, y_train, y_val = train_test_split(
      df['text'], df['intent'], test_size=val_size,
      random_state=SEED, stratify=df['intent']
)

del df

le = LabelEncoder()

tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

def tokenize(texts):
    return tokenizer(
        texts.values.tolist(),
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

x_train_enc = tokenize(x_train)
x_val_enc   = tokenize(x_val)

y_train_enc = torch.from_numpy(le.fit_transform(y_train)).long()
y_val_enc   = torch.from_numpy(le.transform(y_val)).long()

x_val_enc = {k: v.to(device) for k, v in x_val_enc.items()}
y_val_enc = y_val_enc.to(device)

model = IntentClassifier(
    BERT_MODEL_NAME,
    num_classes=len(le.classes_),
    hidden_layer_size=hidden_layer_size,
    dropout_rate=dropout_rate
).to(device)


optimizer = torch.optim.AdamW([
    {"params": model.bert.parameters(),       "lr": bert_lr},
    {"params": model.classifier.parameters(), "lr": head_lr}
], weight_decay=weight_decay)


criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

earlystopping = EarlyStopping(patience=patience)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=scheduler_factor, patience=patience
)

dataset    = TensorDataset(x_train_enc['input_ids'], x_train_enc['attention_mask'], y_train_enc)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct = 0.0, 0

    for batch in dataloader:
        input_ids, attention_mask, labels = [t.to(device) for t in batch]
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        train_correct += (torch.max(outputs, 1).indices == labels).sum().item()

    avg_train_loss = train_loss / len(dataloader)
    train_acc      = train_correct / len(dataset)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}")

    model.eval()
    val_loss, val_acc = EarlyStopping.compute_val_metrics(model, criterion, x_val_enc, y_val_enc)
    print(f"                   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    scheduler.step(val_loss)
    if earlystopping(val_loss, val_acc, model):
        print("Early stopping triggered.")
        break

torch.save(model.state_dict(), f"{MODEL_PATH}bert_fnn.pkl")
print(f"Model saved → {MODEL_PATH}bert_fnn.pkl")
print(f"Best val accuracy: {earlystopping.best_accuracy:.4f}")

architectures.append(model.__class__.__name__)
hyperparams.append(hyper_params_str(BATCH_SIZE, num_epochs, patience, lr, weight_decay, dropout_rate))
scores.append(earlystopping.best_accuracy)
hidden_layers_num .append(1)
hidden_layers_sizes.append(hidden_layer_size)

Original df shape: (765, 2)
Intent distribution:
intent
Make Appointment    139
Bike Types          136
Hours               123
Return Policy       110
Cost Estimation     108
Trade-in Options    100
Welcome Intent       49
Name: count, dtype: int64


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/100 | Train Loss: 1.5698 | Train Acc: 0.4862
                   Val Loss:   0.0088 | Val Acc:   0.7391
Epoch 2/100 | Train Loss: 0.7233 | Train Acc: 0.9015
                   Val Loss:   0.0061 | Val Acc:   0.9217
Epoch 3/100 | Train Loss: 0.5380 | Train Acc: 0.9754
                   Val Loss:   0.0053 | Val Acc:   0.9391
Epoch 4/100 | Train Loss: 0.4812 | Train Acc: 0.9969
                   Val Loss:   0.0050 | Val Acc:   0.9391
Epoch 5/100 | Train Loss: 0.4649 | Train Acc: 1.0000
                   Val Loss:   0.0049 | Val Acc:   0.9391
Epoch 6/100 | Train Loss: 0.4594 | Train Acc: 1.0000
                   Val Loss:   0.0050 | Val Acc:   0.9130
Epoch 7/100 | Train Loss: 0.4570 | Train Acc: 1.0000
                   Val Loss:   0.0051 | Val Acc:   0.9391
Epoch 8/100 | Train Loss: 0.4551 | Train Acc: 1.0000
                   Val Loss:   0.0050 | Val Acc:   0.9391
Epoch 9/100 | Train Loss: 0.4536 | Train Acc: 1.0000
                   Val Loss:   0.0048 | Val Acc:   0.9478
E

Inference using Text CNN

In [3]:
import torch
import numpy as np
import pandas as pd

from tools.cnn import TextCNN
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
from tools.train import train
from tools.earlystopping import EarlyStopping
from tools.preprocessor import NLProcessor
import torch.nn.functional as F


CONFIDENCE_THRESHOLD = 0.86

texts = pd.read_csv(f"{DATA_PATH}intents_test.csv")['text']
model = torch.load(f"{MODEL_PATH}cnn.pkl", weights_only=False)
classes_ = model.classes_

processor = NLProcessor(use_stopwords=True)

X_test = torch.from_numpy(processor.transform(texts).astype(np.float32))

X_test = X_test.permute(0, 2, 1)

with torch.no_grad():

    outputs = model(X_test)
    probs = F.softmax(outputs, dim=1)

max_prob_tensor, predicted_idx = torch.max(probs, dim=1)
max_prob_list   = max_prob_tensor.tolist()
predicted_list  = predicted_idx.tolist()

intents = []
for idx, prob in zip(predicted_list, max_prob_list):
    if prob >= CONFIDENCE_THRESHOLD:
      intents.append(classes_[idx])
    else:
      intents.append("Fallback Intent")

intents_df = pd.DataFrame({
    'text': texts,
    'intent': intents,
})

intents_df.to_csv("intents.csv", index=False)
print(intents_df)

token not found : tiretube
                                                 text            intent
0                     Do I need to replace my helmet?   Fallback Intent
1                My bike was stolen what should I do?   Fallback Intent
2   What types or choices of bikes do I have in yo...   Fallback Intent
3                                  Do you rent bikes?   Fallback Intent
4            I'm a student and I'm doing a project...   Fallback Intent
5                  Can I bring my dog into your shop?   Fallback Intent
6                                Do you match prices?   Cost Estimation
7                  can I get an appointment tomorrow?  Make Appointment
8                         How much to repair my bike?   Cost Estimation
9           How do I know what size tire/tube I need?   Fallback Intent
10                         What's the cost of repair?   Cost Estimation
11                                     Raise the seat   Fallback Intent
12           How long does it take to

Research Diary table

In [10]:
pd.set_option('display.max_colwidth', None)

diary_df = pd.DataFrame({
    'architectures':architectures,
    'hyperparameters':hyperparams,
    'layers':hidden_layers_num,
    'layers_size':hidden_layers_sizes,
    'scores':scores
})
diary_df

,architectures,hyperparameters,layers,layers_size,scores
0,RNN,"batch size : 32, epochs 100, early stopping patience: 8, lr : 0.001, l2 reg: 0.0001, dropout_rate:0.1",2,128,0.878261
1,RNN,"batch size : 32, epochs 120, early stopping patience: 10, lr : 0.001, l2 reg: 0.0001, dropout_rate:0.2",2,128,0.921739
2,RNN,"batch size : 32, epochs 120, early stopping patience: 10, lr : 0.001, l2 reg: 0.0001, dropout_rate:0.2",2,64,0.886957
3,RNN,"batch size : 32, epochs 120, early stopping patience: 10, lr : 0.001, l2 reg: 0.0001, dropout_rate:0.2",2,256,0.921739
4,LSTM,"batch size : 32, epochs 100, early stopping patience: 10, lr : 0.001, l2 reg: 0.0001, dropout_rate:0.2",2,128,0.895652
5,LSTM,"batch size : 32, epochs 100, early stopping patience: 12, lr : 0.001, l2 reg: 0.0001, dropout_rate:0.1",2,128,0.904348
6,LSTM,"batch size : 32, epochs 100, early stopping patience: 12, lr : 0.001, l2 reg: 0.0001, dropout_rate:0.1",2,256,0.939130
7,TextCNN,"batch size : 32, epochs 100, early stopping patience: 10, lr : 0.001, l2 reg: 0.0001, dropout_rate:0.1",2,256,0.939130
8,TextCNN,"batch size : 64, epochs 120, early stopping patience: 12, lr : 0.001, l2 reg: 0.0001, dropout_rate:0.1",2,256,0.956522
9,BERT,"batch size : 32, epochs 100, early stopping patience: 8, lr : 0.0001, l2 reg: 0.0001, dropout_rate:0.1",1,128,0.939148
